In [ ]:
from google.colab import drive
drive.mount('/content/drive')
try:
    import peft
    print('peft already present:', peft.__version__)
except ImportError:
    print('Installing peft...')
    !pip install -q peft

Mounted at /content/drive
peft already present: 0.19.1


In [ ]:
import importlib, sys

report, broken = [], []
for mod in ['numpy', 'PIL', 'torch', 'torchvision', 'timm', 'sklearn', 'scipy', 'pandas', 'peft']:
    try:
        m = importlib.import_module(mod)
        report.append(f'  OK    {mod:14s} {getattr(m, "__version__", "?")}')
    except Exception as e:
        report.append(f'  BROKEN {mod:13s} {type(e).__name__}: {e}')
        broken.append(mod)

print('\n'.join(report))
print(f'\nPython {sys.version.split()[0]}')

if broken:
    print('\n' + '=' * 74)
    print('BROKEN:', ', '.join(broken))
    print('=' * 74)
    if 'sklearn' in broken:
        print('\n(sklearn alone is survivable — section 4 falls back to a PyTorch probe.)')
else:
    print('\nEnvironment clean.')

  OK    numpy          2.0.2
  OK    PIL            11.3.0
  OK    torch          2.11.0+cu128
  OK    torchvision    0.26.0+cu128
  OK    timm           1.0.28
  OK    sklearn        1.6.1
  OK    scipy          1.16.3
  OK    pandas         2.2.2
  OK    peft           0.19.1

Python 3.12.13

Environment clean.


In [ ]:
import os, json, pickle, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import torchvision.models as tvm
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from peft import LoraConfig, get_peft_model
from PIL import Image
from scipy.stats import binomtest

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

PROJECT_DIR  = Path('/content/drive/MyDrive/Indian_Food_Research')
DATASET_PATH = PROJECT_DIR / 'datasets' / 'CompressedDataset'
SPLITS_DIR   = PROJECT_DIR / 'splits'
RESULTS_DIR  = PROJECT_DIR / 'results'
MODELS_DIR   = PROJECT_DIR / 'models'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                              transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                              transforms.RandomHorizontalFlip(), transforms.ColorJitter(0.2, 0.2, 0.2),
                              transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

print('Device:', DEVICE)
for p in (DATASET_PATH, SPLITS_DIR, MODELS_DIR):
    print(('OK   ' if p.exists() else 'MISS '), p)

Device: cuda
OK    /content/drive/MyDrive/Indian_Food_Research/datasets/CompressedDataset
OK    /content/drive/MyDrive/Indian_Food_Research/splits
OK    /content/drive/MyDrive/Indian_Food_Research/models


In [ ]:
full_ds = datasets.ImageFolder(str(DATASET_PATH))
class_names = full_ds.classes
num_classes = len(class_names)

print(f'classes={num_classes}  images={len(full_ds.samples)}')
assert num_classes == 100, f'Expected 100 classes, got {num_classes}. Mount may be mid-sync. STOP.'
assert len(full_ds.samples) == 4578, f'Expected 4578 images, got {len(full_ds.samples)}. STOP.'

train_idx = pickle.load(open(SPLITS_DIR / 'train_idx.pkl', 'rb'))
val_idx   = pickle.load(open(SPLITS_DIR / 'val_idx.pkl',   'rb'))
test_idx  = pickle.load(open(SPLITS_DIR / 'test_idx.pkl',  'rb'))

assert not (set(train_idx) & set(test_idx)), 'train/test overlap'
assert not (set(val_idx)   & set(test_idx)), 'val/test overlap'
assert max(max(train_idx), max(val_idx), max(test_idx)) < len(full_ds.samples)
print(f'train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}  guards passed')


class TransformedSubset(Dataset):
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.transform = base, indices, transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        path, label = self.base.samples[self.indices[i]]
        return self.transform(Image.open(path).convert('RGB')), label


test_loader  = DataLoader(TransformedSubset(full_ds, test_idx, eval_tf),  batch_size=32, num_workers=2)
val_loader   = DataLoader(TransformedSubset(full_ds, val_idx,  eval_tf),  batch_size=32, num_workers=2)
train_loader = DataLoader(TransformedSubset(full_ds, train_idx, train_tf), batch_size=32,
                          shuffle=True, num_workers=2)

y_val_true = np.array([full_ds.targets[i] for i in val_idx])

classes=100  images=4578
train=3204 val=687 test=687  guards passed


In [ ]:
!pip install -q torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.7 MB/s eta 0:00:00


In [ ]:
def build_dinov2(n_classes, pretrained=False):
    return timm.create_model('vit_base_patch14_dinov2', pretrained=pretrained,
                             num_classes=n_classes, img_size=IMG_SIZE)


def build_dinov2_lora(n_classes, r=8, alpha=16, target_modules=('qkv',), dropout=0.1):
    backbone = build_dinov2(n_classes, pretrained=True)
    cfg = LoraConfig(r=r, lora_alpha=alpha, target_modules=list(target_modules),
                     lora_dropout=dropout, modules_to_save=['head'])
    return get_peft_model(backbone, cfg)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    ys, ps = [], []
    for imgs, lbs in loader:
        ps.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy())
        ys.extend(lbs.numpy())
    return np.array(ys), np.array(ps)


preds = {}

m = tvm.resnet50(weights=None)
m.fc = nn.Linear(m.fc.in_features, num_classes)
m.load_state_dict(torch.load(str(MODELS_DIR / 'resnet50_scratch_100pct.pt'), map_location=DEVICE))
y_true, p = predict(m.to(DEVICE), test_loader)
preds['ResNet-50 (scratch)'] = p
del m; torch.cuda.empty_cache()

for label, fname in [('DINOv2 + Linear Probe', 'linear_probe_100pct.pt'),
                     ('DINOv2 + Full FT',      'full_finetune_100pct.pt')]:
    m = build_dinov2(num_classes, pretrained=False)
    m.load_state_dict(torch.load(str(MODELS_DIR / fname), map_location=DEVICE))
    _, p = predict(m.to(DEVICE), test_loader)
    preds[label] = p
    del m; torch.cuda.empty_cache()

m = build_dinov2_lora(num_classes)
m.load_state_dict(torch.load(str(MODELS_DIR / 'dinov2_lora_full.pth'), map_location=DEVICE), strict=True)
_, p = predict(m.to(DEVICE), test_loader)
preds['DINOv2 + LoRA (r=8)'] = p
del m; torch.cuda.empty_cache()

for k, v in preds.items():
    print(f'{k:26s} {100*(v == y_true).mean():.2f}%')

np.save(RESULTS_DIR / 'test_labels.npy', y_true)
for k, v in preds.items():
    np.save(RESULTS_DIR / f"preds_{k.replace(' ', '_').replace('+', '').replace('(', '').replace(')', '').replace('-', '')}.npy", v)

ResNet-50 (scratch)        59.24%
DINOv2 + Linear Probe      94.47%
DINOv2 + Full FT           94.61%
DINOv2 + LoRA (r=8)        95.92%


In [ ]:
def wilson(k, n, z=1.96):
    ph = k / n
    d = 1 + z ** 2 / n
    c = ph + z ** 2 / (2 * n)
    half = z * np.sqrt(ph * (1 - ph) / n + z ** 2 / (4 * n ** 2))
    return 100 * (c - half) / d, 100 * (c + half) / d


n = len(y_true)
rows = []
for name, p in preds.items():
    k = int((p == y_true).sum())
    lo, hi = wilson(k, n)
    rows.append({'Model': name, 'Correct': k, 'n': n, 'Top-1 (%)': round(100 * k / n, 2),
                 '95% CI': f'[{lo:.2f}, {hi:.2f}]', 'CI width (pp)': round(hi - lo, 2)})

ci_df = pd.DataFrame(rows)
ci_df.to_csv(RESULTS_DIR / 'table6_accuracy_with_CI.csv', index=False)
print(ci_df.to_string(index=False))

                Model  Correct   n  Top-1 (%)         95% CI  CI width (pp)
  ResNet-50 (scratch)      407 687      59.24 [55.53, 62.86]           7.33
DINOv2 + Linear Probe      649 687      94.47 [92.50, 95.94]           3.44
     DINOv2 + Full FT      650 687      94.61 [92.66, 96.07]           3.40
  DINOv2 + LoRA (r=8)      659 687      95.92 [94.17, 97.17]           2.99


In [ ]:
names = list(preds)
res = []
for a, b_ in itertools.combinations(names, 2):
    ca, cb = preds[a] == y_true, preds[b_] == y_true
    b = int((ca & ~cb).sum())
    c = int((~ca & cb).sum())
    pval = 1.0 if b + c == 0 else binomtest(b, b + c, 0.5).pvalue
    res.append({'Model A': a, 'Model B': b_, 'A only': b, 'B only': c,
                'Delta (pp)': round(100 * (ca.mean() - cb.mean()), 2),
                'p (exact McNemar)': pval})

mc = pd.DataFrame(res).sort_values('p (exact McNemar)').reset_index(drop=True)
m_tests = len(mc)
raw = mc['p (exact McNemar)'].values
holm = np.minimum.accumulate(
    np.maximum.accumulate((m_tests - np.arange(m_tests)) * raw)[::-1])[::-1].clip(max=1.0)
mc['p (Holm)'] = holm
mc['Significant (a=0.05)'] = np.where(holm < 0.05, 'yes', 'no')
mc['p (exact McNemar)'] = mc['p (exact McNemar)'].round(4)
mc['p (Holm)'] = mc['p (Holm)'].round(4)

mc.to_csv(RESULTS_DIR / 'table6b_mcnemar.csv', index=False)
print(mc.to_string(index=False))

              Model A               Model B  A only  B only  Delta (pp)  p (exact McNemar)  p (Holm) Significant (a=0.05)
  ResNet-50 (scratch)   DINOv2 + LoRA (r=8)       1     253      -36.68             0.0000    0.0000                  yes
  ResNet-50 (scratch)      DINOv2 + Full FT       2     245      -35.37             0.0000    0.0000                  yes
  ResNet-50 (scratch) DINOv2 + Linear Probe       3     245      -35.23             0.0000    0.0000                  yes
DINOv2 + Linear Probe   DINOv2 + LoRA (r=8)       6      16       -1.46             0.0525    0.1574                   no
     DINOv2 + Full FT   DINOv2 + LoRA (r=8)       8      17       -1.31             0.1078    0.2155                   no
DINOv2 + Linear Probe      DINOv2 + Full FT      12      13       -0.15             1.0000    1.0000                   no


In [ ]:
feat_model = timm.create_model('vit_base_patch14_dinov2', pretrained=True,
                               num_classes=0, img_size=IMG_SIZE).to(DEVICE).eval()

@torch.no_grad()
def features(loader):
    F, Y = [], []
    for imgs, lbs in loader:
        F.append(feat_model(imgs.to(DEVICE)).cpu().numpy())
        Y.append(lbs.numpy())
    return np.concatenate(F), np.concatenate(Y)


feat_train_loader = DataLoader(TransformedSubset(full_ds, train_idx, eval_tf),
                               batch_size=64, num_workers=2)
Xtr, ytr = features(feat_train_loader)
Xte, yte = features(test_loader)
print('features:', Xtr.shape, Xte.shape)

del feat_model
torch.cuda.empty_cache()

features: (3204, 768) (687, 768)


In [ ]:
def fit_probe_torch(Xtr, ytr, Xte, C=1.0, iters=500):
    """L2-regularized multinomial logistic regression, L-BFGS. sklearn-free."""
    xtr = torch.tensor(Xtr, dtype=torch.float32, device=DEVICE)
    ytr_t = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
    xte = torch.tensor(Xte, dtype=torch.float32, device=DEVICE)
    lin = nn.Linear(xtr.shape[1], int(ytr.max()) + 1).to(DEVICE)
    wd = 1.0 / (C * len(ytr))
    opt = optim.LBFGS(lin.parameters(), max_iter=iters, history_size=10,
                      line_search_fn='strong_wolfe')
    crit = nn.CrossEntropyLoss()

    def closure():
        opt.zero_grad()
        loss = crit(lin(xtr), ytr_t) + wd * 0.5 * (lin.weight ** 2).sum()
        loss.backward()
        return loss

    opt.step(closure)
    with torch.no_grad():
        return lin(xte).argmax(1).cpu().numpy()


try:
    from sklearn.linear_model import LogisticRegression
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED).fit(Xtr, ytr)
    p_probe = clf.predict(Xte)
    backend = 'sklearn LogisticRegression'
except Exception as e:
    print(f'sklearn unavailable ({type(e).__name__}) — using the PyTorch L-BFGS probe.')
    p_probe = fit_probe_torch(Xtr, ytr, Xte)
    backend = 'PyTorch L-BFGS (sklearn-equivalent)'

k = int((p_probe == yte).sum())
lo, hi = wilson(k, len(yte))
print(f'\nBackend: {backend}')
print(f'Logreg probe @ 224 : {100*k/len(yte):.2f}%  [{lo:.2f}, {hi:.2f}]')
print(f"Trained head @ 224 : {100*(preds['DINOv2 + Linear Probe'] == y_true).mean():.2f}%")
print(f"LoRA r=8    @ 224 : {100*(preds['DINOv2 + LoRA (r=8)'] == y_true).mean():.2f}%")
np.save(RESULTS_DIR / 'preds_probe_logreg_224.npy', np.vstack([yte, p_probe]))

ca = p_probe == yte
cb = preds['DINOv2 + LoRA (r=8)'] == y_true
b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
pv = 1.0 if b + c == 0 else binomtest(b, b + c, 0.5).pvalue
print(f'\nlogreg probe vs LoRA:  b={b}  c={c}  p={pv:.4f}')



Backend: sklearn LogisticRegression
Logreg probe @ 224 : 95.05%  [93.16, 96.44]
Trained head @ 224 : 94.47%
LoRA r=8    @ 224 : 95.92%

logreg probe vs LoRA:  b=6  c=12  p=0.2379


In [ ]:
old = RESULTS_DIR / 'ablation_checkpoints.json'
if old.exists():
    o = pd.DataFrame(json.loads(old.read_text()))
    cols = [c for c in ['config', 'trainable_params', 'trainable_pct', 'val_acc', 'test_acc']
            if c in o.columns]
    print(o[cols].to_string(index=False))
else:
    print('No previous ablation file found.')

      config  trainable_params  trainable_pct  val_acc  test_acc
      r4_qkv            224356       0.261033 0.956332  0.960699
      r8_qkv            371812       0.431854 0.960699  0.947598
     r16_qkv            666724       0.771746 0.962154  0.953421
 r4_qkv+proj            303508       0.352800 0.954876  0.951965
 r8_qkv+proj            530116       0.614591 0.960699  0.954876
r16_qkv+proj            983332       1.134070 0.962154  0.949054


In [ ]:
USE_ZIP = False

if USE_ZIP:
    import shutil
    ZIP = DATASET_PATH.parent / 'CompressedDataset.zip'
    LOCAL = Path('/content/indian_food_cached')
    if not LOCAL.exists():
        assert ZIP.exists(), (
            'Create it once from any cell:\n'
            '!cd "/content/drive/MyDrive/Indian_Food_Research/datasets" && '
            'zip -qr0 CompressedDataset.zip CompressedDataset')
        !cp "{ZIP}" /content/ds.zip && unzip -q /content/ds.zip -d /content/
        shutil.move('/content/CompressedDataset', str(LOCAL))
    ds_for_training = datasets.ImageFolder(str(LOCAL))
    assert ds_for_training.classes == class_names, 'Zip cache class ordering differs from Drive.'
    assert len(ds_for_training.samples) == 4578
    print('Using local zip cache.')
else:
    ds_for_training = full_ds
    print('Training reads directly from Drive.')

train_loader = DataLoader(TransformedSubset(ds_for_training, train_idx, train_tf),
                          batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(TransformedSubset(ds_for_training, val_idx, eval_tf),
                          batch_size=32, num_workers=2)
test_loader  = DataLoader(TransformedSubset(ds_for_training, test_idx, eval_tf),
                          batch_size=32, num_workers=2)

Training reads directly from Drive.


In [ ]:
CKPT = RESULTS_DIR / 'ablation_v2_checkpoints.json'
cur = json.loads(CKPT.read_text()) if CKPT.exists() else []
print([r['config'] for r in cur])

# drop r=4 so it re-runs
cur = [r for r in cur if r['config'] != 'r4_qkv_head']
CKPT.write_text(json.dumps(cur, indent=2))
print('now:', [r['config'] for r in cur])

['r4_qkv_head']
now: []


In [ ]:
# ----------------------------- CELL A: preload -------------------------------
import numpy as np, torch, time
from PIL import Image

PRE_SIZE = 224

def preload(indices, size=PRE_SIZE, label=''):
    buf = torch.empty(len(indices), 3, size, size, dtype=torch.uint8)
    lab = torch.empty(len(indices), dtype=torch.long)
    t0 = time.time()
    for i, idx in enumerate(indices):
        path, y = full_ds.samples[idx]
        im = Image.open(path).convert('RGB').resize((size, size), Image.BILINEAR)
        buf[i] = torch.from_numpy(np.asarray(im).copy()).permute(2, 0, 1)
        lab[i] = y
        if (i + 1) % 500 == 0:
            print(f'  {label} {i+1}/{len(indices)}  ({time.time()-t0:.0f}s)')
    return buf, lab


if 'Xtr_u8' in globals():
    print('Cache already in memory — skipping preload.')
else:
    Xtr_u8, Ytr = preload(train_idx, label='train')
    Xva_u8, Yva = preload(val_idx,   label='val')
    Xte_u8, Yte = preload(test_idx,  label='test')
    print(f'RAM held: {(Xtr_u8.numel()+Xva_u8.numel()+Xte_u8.numel())/1e9:.2f} GB')

assert (Yva.numpy() == y_val_true).all(), 'val label order changed'
assert (Yte.numpy() == y_true).all(),     'test label order changed'
print('Label ordering matches the earlier runs — predictions stay comparable.')

MEAN_T = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1) * 255
STD_T  = torch.tensor(STD,  device=DEVICE).view(1, 3, 1, 1) * 255


def to_input(x_u8, train=False, gen=None):
    x = x_u8.to(DEVICE, non_blocking=True).float()
    if train:
        n = x.size(0)
        flip = torch.rand(n, device=DEVICE, generator=gen) < 0.5
        if flip.any():
            x[flip] = torch.flip(x[flip], dims=[3])
        r = lambda: 1 + (torch.rand(n, 1, 1, 1, device=DEVICE, generator=gen) * 0.4 - 0.2)
        x = x * r()
        g = x.mean(1, keepdim=True)
        x = g + (x - g) * r()
        m = x.mean(dim=(1, 2, 3), keepdim=True)
        x = m + (x - m) * r()
        x = x.clamp_(0, 255)
    return ((x - MEAN_T) / STD_T).contiguous(memory_format=torch.channels_last)


# --------------------------- CELL B: train + ablate --------------------------
import json, itertools
import torch.nn as nn, torch.optim as optim
import pandas as pd

RANKS  = [4,8,16]
EPOCHS = 10
BATCH_CANDIDATES = [64, 48, 32]


@torch.no_grad()
def predict_cached(model, X_u8, bs=128):
    model.eval()
    out = []
    for i in range(0, len(X_u8), bs):
        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            out.append(model(to_input(X_u8[i:i+bs])).argmax(1).cpu())
    return torch.cat(out).numpy()


def run_epoch(model, opt, scaler, crit, gen, batch):
    model.train()
    n = len(Xtr_u8)
    perm = torch.randperm(n)
    for i in range(0, n, batch):
        sel = perm[i:i+batch]
        xb = to_input(Xtr_u8[sel], train=True, gen=gen)
        yb = Ytr[sel].to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            loss = crit(model(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()


def train_model_fast(model, epochs=EPOCHS, lr=1e-3, weight_decay=1e-4):
    model = model.to(DEVICE).to(memory_format=torch.channels_last)
    opt = optim.AdamW([p for p in model.parameters() if p.requires_grad],
                      lr=lr, weight_decay=weight_decay)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
    gen = torch.Generator(device=DEVICE); gen.manual_seed(SEED)
    best_val, best_state, batch = 0.0, None, None

    for ep in range(epochs):
        t0 = time.time()
        for cand in ([batch] if batch else BATCH_CANDIDATES):
            try:
                run_epoch(model, opt, scaler, crit, gen, cand)
                if batch is None:
                    batch = cand
                    print(f'    (batch size {batch})')
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                print(f'    OOM at batch {cand} — retrying smaller')
                if cand == BATCH_CANDIDATES[-1]:
                    raise
        sched.step()
        va = (predict_cached(model, Xva_u8) == y_val_true).mean()
        if va > best_val:
            best_val = va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f'    epoch {ep+1}/{epochs}  val_acc={va:.4f}  ({time.time()-t0:.0f}s)')

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val


CKPT = RESULTS_DIR / 'ablation_v2_checkpoints.json'
abl = json.loads(CKPT.read_text()) if CKPT.exists() else []
done = {r['config'] for r in abl}

for r in RANKS:
    tag = f'r{r}_qkv_head'
    if tag in done:
        print('SKIP', tag, '(already in ablation_v2_checkpoints.json)')
        continue
    print(f'\n=== LoRA {tag} ===')
    model = build_dinov2_lora(num_classes, r=r)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    model, best_val = train_model_fast(model)
    pt = predict_cached(model, Xte_u8)
    k = int((pt == y_true).sum()); lo, hi = wilson(k, len(y_true))
    print(f'  trainable {trainable:,} ({100*trainable/total:.3f}%)  '
          f'test {100*k/len(y_true):.2f}% [{lo:.2f}, {hi:.2f}]')
    abl.append({'config': tag, 'rank': r, 'trainable_params': trainable,
                'trainable_pct': 100 * trainable / total, 'val_acc': float(best_val),
                'test_acc': k / len(y_true), 'ci_low': lo, 'ci_high': hi})
    CKPT.write_text(json.dumps(abl, indent=2))
    np.save(RESULTS_DIR / f'preds_lora_{tag}.npy', pt)
    del model; torch.cuda.empty_cache()

abl_df = pd.DataFrame(abl).sort_values('rank')
abl_df.to_csv(RESULTS_DIR / 'table_lora_rank_ablation_FIXED.csv', index=False)
print('\n', abl_df.to_string(index=False))
print(f'\n{len(abl)}/3 configs done. Change RANKS and re-run for the rest.')

if len(abl) == 3:
    from scipy.stats import binomtest
    print()
    for a, b_ in itertools.combinations([r['config'] for r in abl], 2):
        pa = np.load(RESULTS_DIR / f'preds_lora_{a}.npy')
        pb = np.load(RESULTS_DIR / f'preds_lora_{b_}.npy')
        ca, cb = pa == y_true, pb == y_true
        b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
        pv = 1.0 if b + c == 0 else binomtest(b, b + c, 0.5).pvalue
        print(f'{a} vs {b_}:  delta={100*(ca.mean()-cb.mean()):+.2f} pp  b={b} c={c}  p={pv:.4f}')

Cache already in memory — skipping preload.
Label ordering matches the earlier runs — predictions stay comparable.

=== LoRA r4_qkv_head ===
    (batch size 64)
    epoch 1/10  val_acc=0.9301  (34s)
    epoch 2/10  val_acc=0.9432  (34s)
    epoch 3/10  val_acc=0.9534  (33s)
    epoch 4/10  val_acc=0.9520  (34s)
    epoch 5/10  val_acc=0.9534  (33s)
    epoch 6/10  val_acc=0.9592  (33s)
    epoch 7/10  val_acc=0.9578  (33s)
    epoch 8/10  val_acc=0.9578  (33s)
    epoch 9/10  val_acc=0.9578  (33s)
    epoch 10/10  val_acc=0.9578  (33s)
  trainable 224,356 (0.261%)  test 95.92% [94.17, 97.17]

=== LoRA r8_qkv_head ===
    (batch size 64)
    epoch 1/10  val_acc=0.9243  (33s)
    epoch 2/10  val_acc=0.9491  (33s)
    epoch 3/10  val_acc=0.9418  (33s)
    epoch 4/10  val_acc=0.9578  (33s)
    epoch 5/10  val_acc=0.9636  (33s)
    epoch 6/10  val_acc=0.9592  (33s)
    epoch 7/10  val_acc=0.9651  (33s)
    epoch 8/10  val_acc=0.9651  (33s)
    epoch 9/10  val_acc=0.9636  (33s)
    epoch 10/

In [ ]:
for a, b_ in itertools.combinations([r['config'] for r in abl], 2):
    pa = np.load(RESULTS_DIR / f'preds_lora_{a}.npy')
    pb = np.load(RESULTS_DIR / f'preds_lora_{b_}.npy')
    ca, cb = pa == y_true, pb == y_true
    b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
    pv = 1.0 if b + c == 0 else binomtest(b, b + c, 0.5).pvalue
    print(f'{a} vs {b_}:  delta={100*(ca.mean()-cb.mean()):+.2f} pp  b={b} c={c}  p={pv:.4f}')

r4_qkv_head vs r8_qkv_head:  delta=+0.29 pp  b=5 c=3  p=0.7266
r4_qkv_head vs r16_qkv_head:  delta=+0.73 pp  b=10 c=5  p=0.3018
r8_qkv_head vs r16_qkv_head:  delta=+0.44 pp  b=8 c=5  p=0.5811
